In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, SimpleRNN
from sklearn.ensemble import StackingClassifier
from sklearn import datasets

In [8]:
# prompt: preprocess the pre existed iris dataset from sklearn so i can apply to it svm,lstm,rnn (already the code for the models is written)



# Load the Iris dataset
iris = datasets.load_iris()
X = iris.data
y = iris.target

# Convert to pandas DataFrame for easier handling (optional)
#df = pd.DataFrame(data= np.c_[iris['data'], iris['target']],
#                     columns= iris['feature_names'] + ['target'])


# Scale the features using MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


# Reshape the data for LSTM and RNN (they require 3D input: samples, timesteps, features)
X_train_lstm = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_lstm = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

#Now you can use X_train, X_test, y_train, y_test for SVM
#And X_train_lstm, X_test_lstm, y_train, y_test for LSTM and RNN

In [ ]:
# SVM
svm_model = SVC()
svm_model.fit(X_train, y_train)
svm_predictions = svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_predictions)
print(f"SVM Accuracy: {svm_accuracy}")

In [10]:
# LSTM
lstm_model = Sequential()
lstm_model.add(LSTM(50, activation='relu', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])))
lstm_model.add(Dense(1, activation='sigmoid'))  # Adjust activation based on your problem
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])  # Adjust loss and metrics
lstm_model.fit(X_train_lstm, y_train, epochs=10, batch_size=32, verbose=0) # Adjust epochs and batch_size
lstm_predictions = (lstm_model.predict(X_test_lstm) > 0.5).astype("int32") # Adjust threshold if necessary
lstm_accuracy = accuracy_score(y_test, lstm_predictions)
print(f"LSTM Accuracy: {lstm_accuracy}")

/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 429ms/step
LSTM Accuracy: 0.3


In [11]:
# RNN
rnn_model = Sequential()
rnn_model.add(SimpleRNN(50, activation='relu', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])))
rnn_model.add(Dense(1, activation='sigmoid'))
rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
rnn_model.fit(X_train_lstm, y_train, epochs=10, batch_size=32, verbose=0)
rnn_predictions = (rnn_model.predict(X_test_lstm) > 0.5).astype("int32")
rnn_accuracy = accuracy_score(y_test, rnn_predictions)
print(f"RNN Accuracy: {rnn_accuracy}")


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
RNN Accuracy: 0.3


In [29]:
!pip install scikeras

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
#Import KerasClassifier for wrapping Keras models
# from tensorflow.keras.wrappers.scikit_learn import KerasClassifier # This module is deprecated
from scikeras.wrappers import KerasClassifier # Use scikeras instead
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, SimpleRNN
from sklearn.ensemble import StackingClassifier
from sklearn import datasets

# Function to create the RNN model
def create_rnn_model():
    model = Sequential()
    model.add(SimpleRNN(50, activation='relu', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])))
    model.add(Dense(3, activation='softmax'))  # Output layer with softmax for multi-class
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Function to create the LSTM model
def create_lstm_model():
    model = Sequential()
    model.add(LSTM(50, activation='relu', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])))
    model.add(Dense(3, activation='softmax'))  # Output layer with softmax for multi-class
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Load the Iris dataset
iris = datasets.load_iris()
X = iris.data
y = iris.target

# Scale the features using MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Reshape the data for LSTM and RNN
X_train_lstm = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_lstm = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

# Define the estimators for stacking
#Wrap RNN and LSTM models with KerasClassifier
estimators = [
    ('rnn', KerasClassifier(build_fn=create_rnn_model, epochs=10, batch_size=32, verbose=0)),
    ('lstm', KerasClassifier(build_fn=create_lstm_model, epochs=10, batch_size=32, verbose=0))
]


# Create the stacking classifier
clf = StackingClassifier(estimators=estimators, final_estimator=SVC())

# Reshape data before fitting for RNN/LSTM compatibility
X_train_reshaped = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])


# Fit the stacking classifier using the reshaped data for RNN/LSTM
clf.fit(X_train_reshaped, y_train) # Use reshaped data here

# Make predictions with the stacked model (reshape X_test for RNN/LSTM)
stacking_predictions = clf.predict(X_test_lstm) # Use reshaped data for prediction

# Evaluate the stacked model
stacking_accuracy = accuracy_score(y_test, stacking_predictions)

print(f"Stacking Accuracy: {stacking_accuracy}")


/usr/local/lib/python3.10/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.10/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in 

Stacking Accuracy: 0.7
